In [ ]:
!pip install gymnasium stable-baselines3 numpy pandas matplotlib

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env

class FarmSimulationEnv(gym.Env):
    """
    Custom Environment that follows gymnasium interface.
    Simulates a basic farm with market dynamics and resource allocation.
    """
    metadata = {'render_modes': ['console']}

    def __init__(self):
        super(FarmSimulationEnv, self).__init__()
        
        # Actions: 
        # 0: Do Nothing
        # 1: Plant Seeds (costs money)
        # 2: Water/Fertilize (progresses crop, costs small money)
        # 3: Harvest (moves crop to inventory)
        # 4: Sell Inventory (makes money based on market price)
        self.action_space = spaces.Discrete(5)
        
        # Observation Space:
        # [Cash, Inventory, Crop_Progress, Weather_Condition, Market_Price]
        # Normalized between 0 and 1 for the neural network
        self.observation_space = spaces.Box(
            low=np.array([0, 0, 0, 0, 0]), 
            high=np.array([1, 1, 1, 1, 1]), 
            dtype=np.float32
        )
        
        # Initial states
        self.max_cash = 10000.0
        self.max_inventory = 100.0
        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.cash = 1000.0
        self.inventory = 0.0
        self.crop_progress = 0.0
        self.weather = np.random.uniform(0.2, 1.0) # 1.0 is perfect weather
        self.market_price = np.random.uniform(0.3, 1.0) # Price multiplier
        self.current_step = 0
        self.max_steps = 365 # 1 year simulation
        
        return self._get_obs(), {}

    def _get_obs(self):
        return np.array([
            self.cash / self.max_cash,
            self.inventory / self.max_inventory,
            self.crop_progress,
            self.weather,
            self.market_price
        ], dtype=np.float32)

    def step(self, action):
        self.current_step += 1
        reward = 0
        
        # Environmental changes (Dynamic Market & Weather)
        self.weather = np.clip(self.weather + np.random.normal(0, 0.1), 0.1, 1.0)
        self.market_price = np.clip(self.market_price + np.random.normal(0, 0.05), 0.2, 1.0)
        
        # Action Logic
        if action == 1: # Plant
            if self.cash >= 100 and self.crop_progress == 0:
                self.cash -= 100
                self.crop_progress = 0.1
        
        elif action == 2: # Water/Fertilize
            if self.cash >= 10 and self.crop_progress > 0:
                self.cash -= 10
                # Weather impacts how much progress watering makes
                self.crop_progress += (0.1 * self.weather)
                self.crop_progress = min(1.0, self.crop_progress)
                
        elif action == 3: # Harvest
            if self.crop_progress >= 0.8: # Must be somewhat mature
                yield_amount = self.crop_progress * 50
                if self.inventory + yield_amount <= self.max_inventory:
                    self.inventory += yield_amount
                    self.crop_progress = 0.0 # Reset field
                    
        elif action == 4: # Sell
            if self.inventory > 0:
                revenue = self.inventory * (100 * self.market_price)
                self.cash += revenue
                reward += revenue / 1000.0 # Reward agent for making money
                self.inventory = 0
                
        # Time penalty to encourage efficiency
        reward -= 0.1 
        
        # Terminal conditions
        terminated = bool(self.current_step >= self.max_steps or self.cash <= 0 or self.cash >= self.max_cash)
        truncated = False
        
        # Final reward for remaining cash at end of year
        if terminated and self.cash > 0:
            reward += (self.cash - 1000) / 1000.0
            
        return self._get_obs(), float(reward), terminated, truncated, {}

# Check if environment is valid
env = FarmSimulationEnv()
check_env(env, warn=True)

print("Environment created successfully!")

In [ ]:
# Create environment
env = FarmSimulationEnv()

# Initialize the PPO (Proximal Policy Optimization) agent
print("Starting Training (This might take a minute)...")
model = PPO("MlpPolicy", env, verbose=0, learning_rate=0.001)

# Train the agent
model.learn(total_timesteps=20000)
print("Training Complete!")

# Evaluate the trained agent
obs, info = env.reset()
cash_history = [env.cash]

for _ in range(365):
    # The model predicts the best action based on the observation
    action, _states = model.predict(obs, deterministic=True)
    obs, rewards, terminated, truncated, info = env.step(action)
    cash_history.append(env.cash)
    
    if terminated or truncated:
        break

# Plot the agent's financial performance over the year
plt.figure(figsize=(10, 5))
plt.plot(cash_history, label="Farm Cash Balance", color="green", linewidth=2)
plt.axhline(y=1000, color='r', linestyle='--', label="Starting Balance")
plt.title("AI Agent Performance over 1 Simulated Year")
plt.xlabel("Days")
plt.ylabel("Cash ($)")
plt.legend()
plt.grid(True)
plt.show()

print(f"Final Cash: ${env.cash:.2f}")

In [ ]:
!pip install gradio

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import gradio as gr
from stable_baselines3 import PPO

# ==========================================
# 1. DEFINE THE SIMULATION ENVIRONMENT
# ==========================================
class BetterFarmEnv(gym.Env):
    metadata = {'render_modes': ['console']}

    def __init__(self):
        super(BetterFarmEnv, self).__init__()
        # Actions: 0=Do Nothing, 1=Plant, 2=Water, 3=Harvest, 4=Sell
        self.action_space = spaces.Discrete(5)
        # Observation: [Cash, Inventory, Crop_Progress, Weather, Market_Price]
        self.observation_space = spaces.Box(
            low=np.array([0, 0, 0, 0, 0]), 
            high=np.array([1, 1, 1, 1, 1], dtype=np.float32)
        )
        self.max_cash = 10000.0
        self.max_inventory = 100.0
        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.cash = 1000.0
        self.inventory = 0.0
        self.crop_progress = 0.0
        self.weather = np.random.uniform(0.5, 1.0)
        self.market_price = np.random.uniform(0.5, 1.0)
        self.current_step = 0
        self.max_steps = 365
        return self._get_obs(), {}

    def _get_obs(self):
        return np.array([
            self.cash / self.max_cash,
            self.inventory / self.max_inventory,
            self.crop_progress,
            self.weather,
            self.market_price
        ], dtype=np.float32)

    def step(self, action):
        self.current_step += 1
        reward = 0.0
        
        # Dynamic market and weather shifts
        self.weather = np.clip(self.weather + np.random.normal(0, 0.05), 0.2, 1.0)
        self.market_price = np.clip(self.market_price + np.random.normal(0, 0.05), 0.3, 1.0)
        
        # Action execution with shaped rewards
        if action == 1:  # Plant
            if self.cash >= 100 and self.crop_progress == 0:
                self.cash -= 100
                self.crop_progress = 0.1
                reward += 1.0
                
        elif action == 2:  # Water
            if self.cash >= 10 and 0 < self.crop_progress < 1.0:
                self.cash -= 10
                self.crop_progress += (0.2 * self.weather)
                self.crop_progress = min(1.0, self.crop_progress)
                reward += 0.5
                
        elif action == 3:  # Harvest
            if self.crop_progress >= 0.8:
                yield_amount = self.crop_progress * 50
                if self.inventory + yield_amount <= self.max_inventory:
                    self.inventory += yield_amount
                    self.crop_progress = 0.0
                    reward += 5.0
                    
        elif action == 4:  # Sell
            if self.inventory > 0:
                revenue = self.inventory * (100 * self.market_price)
                self.cash += revenue
                reward += revenue / 50.0
                self.inventory = 0

        terminated = bool(self.current_step >= self.max_steps or self.cash <= 0)
        truncated = False
        
        return self._get_obs(), float(reward), terminated, truncated, {}

# ==========================================
# 2. TRAIN THE REINFORCEMENT LEARNING AGENT
# ==========================================
print("⏳ Initializing environment and training AI agent...")
training_env = BetterFarmEnv()
model = PPO("MlpPolicy", training_env, verbose=0, learning_rate=0.0005)
model.learn(total_timesteps=40000)
print("✅ Training complete! Launching interactive UI...")

# ==========================================
# 3. INTERACTIVE GRADIO DASHBOARD
# ==========================================
ui_env = BetterFarmEnv()
ui_env.reset()

def get_farm_visual(progress):
    if progress == 0: return "🟫 (Empty Dirt)"
    elif progress < 0.4: return "🌱 (Sprouting)"
    elif progress < 0.8: return "🌿 (Growing)"
    else: return "🌾 (Ready to Harvest!)"

def get_inventory_visual(inv):
    boxes = int(inv / 10)
    return "📦" * boxes if boxes > 0 else "Empty"

def take_action(action_name):
    action_map = {
        "Do Nothing ⏳": 0,
        "Plant Seed 🌰 (-$100)": 1,
        "Water Crop 💧 (-$10)": 2,
        "Harvest 🚜": 3,
        "Sell Inventory 💰": 4
    }
    action_num = action_map.get(action_name, 0)
    obs, reward, terminated, truncated, info = ui_env.step(action_num)
    
    cash_str = f"${ui_env.cash:,.2f}"
    weather_str = f"{ui_env.weather*100:.0f}% (100% = optimal)"
    market_str = f"{ui_env.market_price*100:.0f}% (Demand index)"
    farm_vis = get_farm_visual(ui_env.crop_progress)
    inv_vis = f"{ui_env.inventory:.0f} units | {get_inventory_visual(ui_env.inventory)}"
    log = f"Day {ui_env.current_step}/365: Action taken: '{action_name}'. Reward: +{reward:.2f}"
    
    if terminated:
        log += f"\n🏆 YEAR COMPLETED! Final Cash: ${ui_env.cash:,.2f}. Resetting farm for new season..."
        ui_env.reset()
        
    return cash_str, weather_str, market_str, farm_vis, inv_vis, log

def ai_step():
    obs = ui_env._get_obs()
    action, _ = model.predict(obs, deterministic=True)
    action_names = [
        "Do Nothing ⏳", 
        "Plant Seed 🌰 (-$100)", 
        "Water Crop 💧 (-$10)", 
        "Harvest 🚜", 
        "Sell Inventory 💰"
    ]
    return take_action(action_names[action])

# Build Layout
with gr.Blocks(theme=gr.themes.Soft(), title="AI Farm Simulator") as demo:
    gr.Markdown("# 🚜 Autonomous AI Farming Agent Simulation")
    gr.Markdown("Test the trained PPO Reinforcement Learning model in a dynamic market environment.")
    
    with gr.Row():
        with gr.Column(scale=2):
            cash_out = gr.Textbox(label="💰 Bank Balance", value="$1,000.00")
            with gr.Row():
                weather_out = gr.Textbox(label="☀️ Weather Quality", value="Starting...")
                market_out = gr.Textbox(label="📈 Market Price Index", value="Starting...")
            
            gr.Markdown("### 🌾 Crop Field Status")
            farm_out = gr.Textbox(label="Field Progress", value="🟫 (Empty Dirt)")
            
            gr.Markdown("### 🛖 Storage Barn")
            inv_out = gr.Textbox(label="Crop Inventory", value="Empty")
            
        with gr.Column(scale=1):
            gr.Markdown("### 🎮 Manual Actions")
            btn_plant = gr.Button("Plant Seed 🌰 (-$100)", variant="primary")
            btn_water = gr.Button("Water Crop 💧 (-$10)", variant="secondary")
            btn_harvest = gr.Button("Harvest 🚜", variant="primary")
            btn_sell = gr.Button("Sell Inventory 💰", variant="secondary")
            btn_wait = gr.Button("Do Nothing ⏳")
            
            gr.Markdown("### 🤖 Autonomous AI Control")
            btn_ai = gr.Button("Let AI Take 1 Step 🧠", variant="stop")
            
    log_out = gr.Textbox(label="Activity & Financial Log", lines=3)

    outputs = [cash_out, weather_out, market_out, farm_out, inv_out, log_out]
    
    btn_plant.click(fn=lambda: take_action("Plant Seed 🌰 (-$100)"), outputs=outputs)
    btn_water.click(fn=lambda: take_action("Water Crop 💧 (-$10)"), outputs=outputs)
    btn_harvest.click(fn=lambda: take_action("Harvest 🚜"), outputs=outputs)
    btn_sell.click(fn=lambda: take_action("Sell Inventory 💰"), outputs=outputs)
    btn_wait.click(fn=lambda: take_action("Do Nothing ⏳"), outputs=outputs)
    btn_ai.click(fn=ai_step, outputs=outputs)

# ==========================================
# 4. LAUNCH WITH PUBLIC LINK
# ==========================================
demo.launch(inline=True, share=True)

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import gradio as gr
from stable_baselines3 import PPO

# ==========================================
# 1. SHARED MULTI-AGENT MANDI ENVIRONMENT
# ==========================================
class MultiAgentFarmEnv:
    def __init__(self):
        self.max_steps = 365
        self.reset()

    def reset(self):
        self.current_step = 0
        self.monsoon = np.random.uniform(0.6, 1.0)
        self.base_demand = np.random.uniform(0.6, 1.0)
        
        # State for each agent: [cash, inventory (quintals), crop_progress]
        self.agents = {
            "Agent_1 (Your AI)": {"cash": 100000.0, "inventory": 0.0, "crop": 0.0},
            "Agent_2 (Rival AI)": {"cash": 100000.0, "inventory": 0.0, "crop": 0.0}
        }
        return self.get_states()

    def get_states(self):
        return {
            name: {
                "cash": data["cash"],
                "inventory": data["inventory"],
                "crop": data["crop"],
                "monsoon": self.monsoon,
                "market_demand": self.base_demand
            }
            for name, data in self.agents.items()
        }

    def step(self, action_1, action_2):
        """
        Actions: 0=Wait, 1=Plant (-₹10k), 2=Irrigate (-₹2k), 3=Harvest, 4=Sell
        """
        self.current_step += 1
        
        # Dynamic climate shifts
        self.monsoon = float(np.clip(self.monsoon + np.random.normal(0, 0.04), 0.2, 1.0))
        self.base_demand = float(np.clip(self.base_demand + np.random.normal(0, 0.05), 0.3, 1.0))

        actions = {"Agent_1 (Your AI)": action_1, "Agent_2 (Rival AI)": action_2}
        total_supply_sold = 0.0

        # Pass 1: Handle planting, irrigating, and harvesting
        for name, act in actions.items():
            agent = self.agents[name]
            if act == 1 and agent["cash"] >= 10000 and agent["crop"] == 0:
                agent["cash"] -= 10000
                agent["crop"] = 0.1
            elif act == 2 and agent["cash"] >= 2000 and 0 < agent["crop"] < 1.0:
                agent["cash"] -= 2000
                agent["crop"] = min(1.0, agent["crop"] + (0.2 * self.monsoon))
            elif act == 3 and agent["crop"] >= 0.8:
                agent["inventory"] += agent["crop"] * 100  # Quintals
                agent["crop"] = 0.0
            elif act == 4 and agent["inventory"] > 0:
                total_supply_sold += agent["inventory"]

        # Pass 2: Shared Market Price Calculation (Supply & Demand dynamics)
        # Oversupply penalty: price drops up to 40% if huge volume is dumped at once
        oversupply_penalty = min(0.40, (total_supply_sold / 200.0) * 0.40)
        effective_price = max(800.0, 2500.0 * (self.base_demand - oversupply_penalty))

        # Pass 3: Process sales at the unified clearing price
        for name, act in actions.items():
            agent = self.agents[name]
            if act == 4 and agent["inventory"] > 0:
                revenue = agent["inventory"] * effective_price
                agent["cash"] += revenue
                agent["inventory"] = 0.0

        done = self.current_step >= self.max_steps
        return self.get_states(), effective_price, total_supply_sold, done

# ==========================================
# 2. DECISION ENGINES (RL + RIVAL STRATEGY)
# ==========================================
env = MultiAgentFarmEnv()

def rival_ai_policy(state):
    """Competitive heuristic agent designed to time the market."""
    if state["crop"] == 0.0 and state["cash"] >= 10000:
        return 1  # Plant
    elif 0 < state["crop"] < 0.8 and state["cash"] >= 2000:
        return 2  # Irrigate
    elif state["crop"] >= 0.8:
        return 3  # Harvest
    elif state["inventory"] > 0 and state["market_demand"] >= 0.65:
        return 4  # Sell when market demand is high
    return 0  # Hold / Wait

def format_crop_stage(crop_val):
    if crop_val == 0: return "🟫 Barren"
    elif crop_val < 0.4: return "🌱 Sowing"
    elif crop_val < 0.8: return "🌿 Vegetative"
    else: return "🌾 Ready (100%)"

# ==========================================
# 3. MULTI-AGENT INTERACTIVE DASHBOARD
# ==========================================
def simulate_step(manual_action_agent1=None):
    states = env.get_states()
    
    # Agent 1 decision: manual override or automated RL policy
    if manual_action_agent1 is not None:
        act1 = manual_action_agent1
    else:
        # Default AI rule logic for Agent 1 if on auto-step
        s1 = states["Agent_1 (Your AI)"]
        act1 = 4 if s1["inventory"] > 0 and s1["market_demand"] > 0.6 else (3 if s1["crop"] >= 0.8 else (2 if s1["crop"] > 0 else 1))

    # Agent 2 (Rival AI) decision
    act2 = rival_ai_policy(states["Agent_2 (Rival AI)"])

    action_labels = ["Wait ⏳", "Plant 🌰", "Irrigate 💧", "Harvest 🚜", "Sell 💰"]
    new_states, clearing_price, total_sold, done = env.step(act1, act2)

    a1 = new_states["Agent_1 (Your AI)"]
    a2 = new_states["Agent_2 (Rival AI)"]

    market_log = (
        f"Day {env.current_step}/365 | Clearing Price: ₹{clearing_price:,.2f}/Q | "
        f"Total Mandi Volume: {total_sold:.0f} Q\n"
        f"• Agent 1 did: {action_labels[act1]} | Cash: ₹{a1['cash']:,.2f} | Godown: {a1['inventory']:.0f} Q\n"
        f"• Agent 2 did: {action_labels[act2]} | Cash: ₹{a2['cash']:,.2f} | Godown: {a2['inventory']:.0f} Q"
    )

    if done:
        winner = "Agent 1 (Your AI)" if a1['cash'] > a2['cash'] else "Agent 2 (Rival AI)"
        market_log += f"\n\n🏆 SEASON OVER! Winner: {winner} with ₹{max(a1['cash'], a2['cash']):,.2f}!"
        env.reset()

    return (
        f"₹{a1['cash']:,.2f}", f"{a1['inventory']:.0f} Q", format_crop_stage(a1['crop']),
        f"₹{a2['cash']:,.2f}", f"{a2['inventory']:.0f} Q", format_crop_stage(a2['crop']),
        f"{a1['monsoon']*100:.0f}%", f"₹{clearing_price:,.2f} / Quintal",
        market_log
    )

def reset_season():
    env.reset()
    return simulate_step(0)

# Dashboard Layout
with gr.Blocks() as demo:
    gr.Markdown("# 🌾 Multi-Agent Agricultural Economy Simulation")
    gr.Markdown("Two autonomous farming agents competing in a shared Mandi market with dynamic supply-demand price discovery.")

    with gr.Row():
        monsoon_box = gr.Textbox(label="🌧️ Current Monsoon Index", value="80%")
        price_box = gr.Textbox(label="📈 Mandi Spot Price", value="₹2,500.00 / Quintal")

    with gr.Row():
        # Agent 1 Column
        with gr.Column(variant="panel"):
            gr.Markdown("### 🤖 Agent 1: Primary AI Farm")
            a1_cash = gr.Textbox(label="Bank Balance", value="₹1,00,000.00")
            a1_inv = gr.Textbox(label="Godown Inventory", value="0 Q")
            a1_crop = gr.Textbox(label="Field Stage", value="🟫 Barren")
            
            gr.Markdown("**Manual Override Controls:**")
            with gr.Row():
                btn_p1 = gr.Button("Plant 🌰", size="sm")
                btn_w1 = gr.Button("Irrigate 💧", size="sm")
                btn_h1 = gr.Button("Harvest 🚜", size="sm")
                btn_s1 = gr.Button("Sell at Mandi 💰", size="sm", variant="primary")

        # Agent 2 Column
        with gr.Column(variant="panel"):
            gr.Markdown("### 🥊 Agent 2: Rival AI Farm")
            a2_cash = gr.Textbox(label="Bank Balance", value="₹1,00,000.00")
            a2_inv = gr.Textbox(label="Godown Inventory", value="0 Q")
            a2_crop = gr.Textbox(label="Field Stage", value="🟫 Barren")
            gr.Markdown("*(Operates autonomously via market-timing heuristic policy)*")

    with gr.Row():
        btn_auto_step = gr.Button("▶️ Simulate 1 Day (Both Auto)", variant="secondary")
        btn_reset = gr.Button("🔄 Reset Season", variant="stop")

    log_box = gr.Textbox(label="📊 Live Market & Transaction Feed", lines=4)

    outputs = [a1_cash, a1_inv, a1_crop, a2_cash, a2_inv, a2_crop, monsoon_box, price_box, log_box]

    # Wiring actions
    btn_p1.click(fn=lambda: simulate_step(1), outputs=outputs)
    btn_w1.click(fn=lambda: simulate_step(2), outputs=outputs)
    btn_h1.click(fn=lambda: simulate_step(3), outputs=outputs)
    btn_s1.click(fn=lambda: simulate_step(4), outputs=outputs)
    btn_auto_step.click(fn=lambda: simulate_step(None), outputs=outputs)
    btn_reset.click(fn=reset_season, outputs=outputs)

demo.launch(inline=True, share=False, theme=gr.themes.Soft())

In [ ]:
!pip install langchain-google-genai langchain

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import gradio as gr
import os
from langchain_google_genai import ChatGoogleGenerativeAI

# ==========================================
# 0. SETUP LANGCHAIN LLM
# ==========================================
# Put your API key here
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY", "")  # Set GOOGLE_API_KEY in your environment; never hardcode secrets.
try:
    llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.3)
except Exception as e:
    print("LLM Setup failed. Make sure your API key is correct.")
    llm = None

# ==========================================
# 1. MULTI-CROP ENVIRONMENT
# ==========================================
CROP_DATA = {
    0: {"name": "Barren", "cost": 0, "water_need": 0, "growth_rate": 0, "yield": 0},
    1: {"name": "Wheat 🌾", "cost": 10000, "water_need": 0.2, "growth_rate": 0.20, "yield": 100},
    2: {"name": "Mustard 🌼", "cost": 5000, "water_need": 0.1, "growth_rate": 0.35, "yield": 60},
    3: {"name": "Sugarcane 🎋", "cost": 25000, "water_need": 0.4, "growth_rate": 0.10, "yield": 300},
}

class AdvancedMultiAgentEnv:
    def __init__(self):
        self.max_steps = 365
        self.reset()

    def reset(self):
        self.current_step = 0
        self.monsoon = np.random.uniform(0.6, 1.0)
        self.base_demand = np.random.uniform(0.6, 1.0)
        self.market_history = []
        
        # State: cash, inventory_amt, active_crop_type, crop_progress
        self.agents = {
            "Agent_1": {"cash": 100000.0, "inv": 0.0, "crop_type": 0, "crop_prog": 0.0},
            "Agent_2": {"cash": 100000.0, "inv": 0.0, "crop_type": 0, "crop_prog": 0.0}
        }
        return self.agents

    def step(self, act1, act2):
        """
        Actions: 0=Wait, 1=Plant Wheat, 2=Plant Mustard, 3=Plant Sugarcane, 4=Irrigate, 5=Harvest, 6=Sell
        """
        self.current_step += 1
        self.monsoon = float(np.clip(self.monsoon + np.random.normal(0, 0.04), 0.2, 1.0))
        self.base_demand = float(np.clip(self.base_demand + np.random.normal(0, 0.05), 0.3, 1.0))

        actions = {"Agent_1": act1, "Agent_2": act2}
        total_supply_sold = 0.0

        for name, act in actions.items():
            a = self.agents[name]
            
            # Planting (1=Wheat, 2=Mustard, 3=Sugarcane)
            if act in [1, 2, 3] and a["crop_type"] == 0:
                cost = CROP_DATA[act]["cost"]
                if a["cash"] >= cost:
                    a["cash"] -= cost
                    a["crop_type"] = act
                    a["crop_prog"] = 0.1
                    
            # Irrigate (4)
            elif act == 4 and a["crop_type"] != 0 and a["crop_prog"] < 1.0:
                cost = 2000
                if a["cash"] >= cost:
                    a["cash"] -= cost
                    # Growth depends on monsoon vs crop's water need
                    water_efficiency = self.monsoon / CROP_DATA[a["crop_type"]]["water_need"]
                    actual_growth = CROP_DATA[a["crop_type"]]["growth_rate"] * min(1.5, water_efficiency)
                    a["crop_prog"] = min(1.0, a["crop_prog"] + actual_growth)
                    
            # Harvest (5)
            elif act == 5 and a["crop_prog"] >= 0.8:
                a["inv"] += CROP_DATA[a["crop_type"]]["yield"]
                a["crop_type"] = 0
                a["crop_prog"] = 0.0
                
            # Sell (6)
            elif act == 6 and a["inv"] > 0:
                total_supply_sold += a["inv"]

        # Market Pricing
        penalty = min(0.40, (total_supply_sold / 200.0) * 0.40)
        price = max(1000.0, 2500.0 * (self.base_demand - penalty))

        for name, act in actions.items():
            a = self.agents[name]
            if act == 6 and a["inv"] > 0:
                revenue = a["inv"] * price
                a["cash"] += revenue
                a["inv"] = 0.0
                self.market_history.append(f"Day {self.current_step}: {name} sold at ₹{price:.0f}/Q. Balance: ₹{a['cash']:.0f}")

        done = self.current_step >= self.max_steps
        return self.agents, price, done

env = AdvancedMultiAgentEnv()

def rival_policy(s):
    # Agent 2 Strategy: Prefers fast/cheap Mustard to undercut market
    if s["crop_type"] == 0 and s["cash"] >= 5000: return 2 
    elif 0 < s["crop_prog"] < 0.8 and s["cash"] >= 2000: return 4
    elif s["crop_prog"] >= 0.8: return 5
    elif s["inv"] > 0: return 6
    return 0

# ==========================================
# 2. GRADIO DASHBOARD & LLM INTEGRATION
# ==========================================
def get_stage(c_type, prog):
    if c_type == 0: return "🟫 Barren"
    name = CROP_DATA[c_type]["name"]
    return f"{name} ({prog*100:.0f}%)"

def generate_llm_summary(history_text, a1_cash, a2_cash):
    if not llm: return "LLM not configured. Please add your API key."
    prompt = f"""
    You are an agricultural economist. Review this simulated 1-year market data.
    Agent 1 Final Cash: ₹{a1_cash}
    Agent 2 Final Cash: ₹{a2_cash}
    
    Transaction Log:
    {history_text}
    
    Write a brief, 3-sentence executive summary explaining why the winning agent made more money based on their market timing and sales.
    """
    try:
        return llm.invoke(prompt).content
    except Exception as e:
        return "LLM Analysis failed. Check API key."

def sim_step(action1):
    a1_state = env.agents["Agent_1"]
    
    # Auto-policy fallback for Agent 1 (Prefers Sugarcane if rich)
    if action1 is None:
        if a1_state["inv"] > 0: action1 = 6
        elif a1_state["crop_prog"] >= 0.8: action1 = 5
        elif a1_state["crop_prog"] > 0: action1 = 4
        elif a1_state["crop_type"] == 0:
            action1 = 3 if a1_state["cash"] >= 25000 else 1
        else: action1 = 0

    action2 = rival_policy(env.agents["Agent_2"])
    
    states, price, done = env.step(action1, action2)
    s1, s2 = states["Agent_1"], states["Agent_2"]
    
    log = f"Day {env.current_step}/365 | Spot Price: ₹{price:,.2f}/Q"
    analysis = "Season in progress..."

    if done:
        log += f"\n\n🏆 SEASON OVER! A1: ₹{s1['cash']:,.0f} | A2: ₹{s2['cash']:,.0f}"
        hist = "\n".join(env.market_history[-10:]) # Send last 10 major trades to LLM
        analysis = generate_llm_summary(hist, s1['cash'], s2['cash'])
        env.reset()

    return (
        f"₹{s1['cash']:,.2f}", f"{s1['inv']:.0f} Q", get_stage(s1['crop_type'], s1['crop_prog']),
        f"₹{s2['cash']:,.2f}", f"{s2['inv']:.0f} Q", get_stage(s2['crop_type'], s2['crop_prog']),
        f"{env.monsoon*100:.0f}%", f"₹{price:,.2f}", log, analysis
    )

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🌱 Advanced Multi-Crop Economy with AI Market Analysis")
    
    with gr.Row():
        monsoon_box = gr.Textbox(label="🌧️ Monsoon Index")
        price_box = gr.Textbox(label="📈 Mandi Spot Price")

    with gr.Row():
        with gr.Column(variant="panel"):
            gr.Markdown("### 🤖 Agent 1 (Your Farm)")
            a1_c = gr.Textbox(label="Balance", value="₹1,00,000")
            a1_i = gr.Textbox(label="Inventory", value="0 Q")
            a1_p = gr.Textbox(label="Field", value="🟫 Barren")
            
            with gr.Row():
                b_w = gr.Button("Wheat 🌾 (-₹10k)", size="sm")
                b_m = gr.Button("Mustard 🌼 (-₹5k)", size="sm")
                b_s = gr.Button("Sugarcane 🎋 (-₹25k)", size="sm")
            with gr.Row():
                b_irr = gr.Button("Irrigate 💧", size="sm")
                b_harv = gr.Button("Harvest 🚜", size="sm")
                b_sell = gr.Button("Sell 💰", size="sm", variant="primary")

        with gr.Column(variant="panel"):
            gr.Markdown("### 🥊 Agent 2 (Rival AI)")
            a2_c = gr.Textbox(label="Balance", value="₹1,00,000")
            a2_i = gr.Textbox(label="Inventory", value="0 Q")
            a2_p = gr.Textbox(label="Field", value="🟫 Barren")
            gr.Markdown("*Agent 2 dynamically plants Mustard to flood the market.*")

    with gr.Row():
        b_auto = gr.Button("▶️ Simulate 1 Day (Auto)", variant="secondary")
        
    with gr.Row():
        log_box = gr.Textbox(label="Live Market Feed")
        llm_box = gr.Textbox(label="🧠 LLM Post-Season Analysis", lines=4)

    outs = [a1_c, a1_i, a1_p, a2_c, a2_i, a2_p, monsoon_box, price_box, log_box, llm_box]
    
    b_w.click(lambda: sim_step(1), outputs=outs)
    b_m.click(lambda: sim_step(2), outputs=outs)
    b_s.click(lambda: sim_step(3), outputs=outs)
    b_irr.click(lambda: sim_step(4), outputs=outs)
    b_harv.click(lambda: sim_step(5), outputs=outs)
    b_sell.click(lambda: sim_step(6), outputs=outs)
    b_auto.click(lambda: sim_step(None), outputs=outs)

demo.launch(inline=True, share=False)

In [ ]:
%pip install pyngrok

In [ ]:
import gymnasium as gym
import numpy as np
import gradio as gr
import os
from langchain_google_genai import ChatGoogleGenerativeAI

# ==========================================
# 0. SETUP LLM & ENVIRONMENT CONSTANTS
# ==========================================
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY", "")  # Set GOOGLE_API_KEY in your environment; never hardcode secrets.
try:
    llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.3)
except Exception:
    llm = None

CROP_DATA = {
    0: {"name": "Barren", "cost": 0, "water_need": 0, "growth_rate": 0, "yield": 0},
    1: {"name": "Wheat 🌾", "cost": 10000, "water_need": 0.2, "growth_rate": 0.20, "yield": 100},
    2: {"name": "Mustard 🌼", "cost": 5000, "water_need": 0.1, "growth_rate": 0.35, "yield": 60},
    3: {"name": "Sugarcane 🎋", "cost": 25000, "water_need": 0.4, "growth_rate": 0.10, "yield": 300},
}

# ==========================================
# 1. 7-DAY TIME-SERIES MARKET PREDICTOR
# ==========================================
class MandiMarketPredictor:
    """Rolling-window forecasting engine for Mandi spot prices."""
    def __init__(self, history_window=14):
        self.history_window = history_window
        self.price_history = []

    def record_price(self, price):
        self.price_history.append(price)
        if len(self.price_history) > 60:
            self.price_history.pop(0)

    def forecast_7_days(self, current_price, monsoon_index):
        if len(self.price_history) < 3:
            # Fallback baseline when starting
            return [current_price * (1.0 + 0.01 * i) for i in range(1, 8)], "NEUTRAL"

        # Calculate momentum slope over the recent window
        recent = self.price_history[-min(len(self.price_history), self.history_window):]
        x = np.arange(len(recent))
        slope, _ = np.polyfit(x, recent, 1)

        # 7-day projections combining drift, monsoon factor, and mean reversion
        forecasts = []
        base_target = 2500.0 * monsoon_index
        for day in range(1, 8):
            trend_val = current_price + (slope * day)
            reversion = 0.05 * (base_target - trend_val)
            pred = max(900.0, trend_val + reversion)
            forecasts.append(round(float(pred), 2))

        # Generate trading signal
        max_future_price = max(forecasts)
        if max_future_price > current_price * 1.08:
            signal = "🟢 BULLISH (Hold for higher Mandi prices)"
        elif min(forecasts) < current_price * 0.93:
            signal = "🔴 BEARISH (Sell now; price drop expected)"
        else:
            signal = "🟡 NEUTRAL (Market stable)"

        return forecasts, signal

# ==========================================
# 2. FULL MULTI-AGENT ENVIRONMENT
# ==========================================
class FullFarmEnv:
    def __init__(self):
        self.max_steps = 365
        self.predictor = MandiMarketPredictor()
        self.reset()

    def reset(self):
        self.current_step = 0
        self.monsoon = float(np.random.uniform(0.6, 1.0))
        self.base_demand = float(np.random.uniform(0.6, 1.0))
        self.market_history = []
        self.current_price = 2200.0
        self.predictor.price_history = []
        self.predictor.record_price(self.current_price)
        
        self.agents = {
            "Agent_1": {"cash": 100000.0, "inv": 0.0, "crop_type": 0, "crop_prog": 0.0, "cattle": 0, "labor": 0},
            "Agent_2": {"cash": 100000.0, "inv": 0.0, "crop_type": 0, "crop_prog": 0.0, "cattle": 0, "labor": 0}
        }
        return self.agents

    def step(self, act1, act2):
        self.current_step += 1
        self.monsoon = float(np.clip(self.monsoon + np.random.normal(0, 0.04), 0.2, 1.0))
        self.base_demand = float(np.clip(self.base_demand + np.random.normal(0, 0.05), 0.3, 1.0))
        actions = {"Agent_1": act1, "Agent_2": act2}
        total_supply_sold = 0.0

        for name, act in actions.items():
            a = self.agents[name]
            
            # Daily Passive Operations (Labor & Cattle)
            if a["labor"] > 0:
                if a["cash"] >= 200: a["cash"] -= 200
                else: a["labor"] = 0
            
            if a["cattle"] > 0:
                feed = a["cattle"] * 150
                milk = a["cattle"] * 400
                if a["cash"] >= feed:
                    a["cash"] += (milk - feed)
                else:
                    a["cattle"] = max(0, a["cattle"] - 1)

            # Active Decisions
            if act in [1, 2, 3] and a["crop_type"] == 0:
                cost = CROP_DATA[act]["cost"]
                if a["cash"] >= cost:
                    a["cash"] -= cost
                    a["crop_type"] = act
                    a["crop_prog"] = 0.1
                    
            elif act == 4 and a["crop_type"] != 0 and a["crop_prog"] < 1.0:
                if a["cash"] >= 2000:
                    a["cash"] -= 2000
                    efficiency = self.monsoon / CROP_DATA[a["crop_type"]]["water_need"]
                    mult = 1.5 if a["labor"] > 0 else 1.0
                    a["crop_prog"] = min(1.0, a["crop_prog"] + CROP_DATA[a["crop_type"]]["growth_rate"] * min(1.5, efficiency) * mult)
                    
            elif act == 5 and a["crop_prog"] >= 0.8:
                a["inv"] += CROP_DATA[a["crop_type"]]["yield"]
                a["crop_type"] = 0
                a["crop_prog"] = 0.0
                
            elif act == 6 and a["inv"] > 0:
                total_supply_sold += a["inv"]
                
            elif act == 7 and a["cash"] >= 40000:
                a["cash"] -= 40000
                a["cattle"] += 1
                self.market_history.append(f"Day {self.current_step}: {name} bought 1 Cow.")
                
            elif act == 8 and a["labor"] == 0:
                a["labor"] = 1
                self.market_history.append(f"Day {self.current_step}: {name} hired farm labor.")

        # Price Clearing
        oversupply = min(0.40, (total_supply_sold / 200.0) * 0.40)
        self.current_price = max(900.0, float(2500.0 * (self.base_demand - oversupply)))
        self.predictor.record_price(self.current_price)

        for name, act in actions.items():
            a = self.agents[name]
            if act == 6 and a["inv"] > 0:
                a["cash"] += (a["inv"] * self.current_price)
                self.market_history.append(f"Day {self.current_step}: {name} sold {a['inv']:.0f}Q at ₹{self.current_price:.0f}/Q.")
                a["inv"] = 0.0

        done = self.current_step >= self.max_steps
        return self.agents, self.current_price, done

env = FullFarmEnv()

# ==========================================
# 3. FORECAST-INFORMED AGENT POLICIES
# ==========================================
def rival_policy(s, signal):
    # Agent 2 uses market signal for selling
    if s["inv"] > 0 and "BEARISH" in signal:
        return 6 # Dump immediately if price crash predicted
    elif s["inv"] > 0 and "BULLISH" not in signal:
        return 6
    elif s["cattle"] == 0 and s["cash"] >= 45000: return 7 
    elif s["crop_type"] == 0 and s["cash"] >= 10000: return 1
    elif 0 < s["crop_prog"] < 0.8 and s["cash"] >= 2000: return 4
    elif s["crop_prog"] >= 0.8: return 5
    return 0

def format_stage(ctype, prog):
    if ctype == 0: return "🟫 Barren"
    return f"{CROP_DATA[ctype]['name']} ({prog*100:.0f}%)"

def get_llm_summary(history, c1, c2):
    if not llm: return "LLM not configured."
    prompt = f"""
    Review this 1-year agricultural market simulation:
    Agent 1 Balance: ₹{c1} | Agent 2 Balance: ₹{c2}
    Transaction Log: {history}
    
    Write a 3-sentence economic breakdown explaining how the 7-day Mandi price predictions, crop selection, and livestock investments influenced the final winner.
    """
    try:
        return llm.invoke(prompt).content
    except Exception:
        return "LLM Analysis unavailable."

# ==========================================
# 4. SIMULATION STEP & DASHBOARD
# ==========================================
def sim_step(act1):
    a1 = env.agents["Agent_1"]
    forecasts, signal = env.predictor.forecast_7_days(env.current_price, env.monsoon)
    
    # Auto-policy for Agent 1: Uses 7-day predictive timing for selling
    if act1 is None:
        if a1["inv"] > 0:
            act1 = 0 if "BULLISH" in signal else 6 # Hold if bullish, sell if bearish/neutral
        elif a1["cattle"] < 2 and a1["cash"] > 50000: act1 = 7
        elif a1["labor"] == 0 and a1["cash"] > 20000: act1 = 8
        elif a1["crop_prog"] >= 0.8: act1 = 5
        elif a1["crop_prog"] > 0: act1 = 4
        elif a1["crop_type"] == 0: act1 = 3 if a1["cash"] > 30000 else 2
        else: act1 = 0

    act2 = rival_policy(env.agents["Agent_2"], signal)
    states, price, done = env.step(act1, act2)
    s1, s2 = states["Agent_1"], states["Agent_2"]
    
    # Re-compute forecast after current day's price
    forecasts, signal = env.predictor.forecast_7_days(price, env.monsoon)
    forecast_str = " | ".join([f"D+{i+1}: ₹{p:.0f}" for i, p in enumerate(forecasts)])

    log = f"Day {env.current_step}/365 | Spot Price: ₹{price:,.2f}/Q"
    analysis = "Season in progress..."

    if done:
        log += f"\n\n🏆 SEASON COMPLETED! Agent 1: ₹{s1['cash']:,.0f} | Agent 2: ₹{s2['cash']:,.0f}"
        hist = "\n".join(env.market_history[-12:])
        analysis = get_llm_summary(hist, s1['cash'], s2['cash'])
        env.reset()

    return (
        f"₹{s1['cash']:,.2f}", f"{s1['inv']:.0f} Q", format_stage(s1['crop_type'], s1['crop_prog']), f"{s1['cattle']} 🐄", f"{'Yes 👨‍🌾' if s1['labor'] else 'No'}",
        f"₹{s2['cash']:,.2f}", f"{s2['inv']:.0f} Q", format_stage(s2['crop_type'], s2['crop_prog']), f"{s2['cattle']} 🐄", f"{'Yes 👨‍🌾' if s2['labor'] else 'No'}",
        f"{env.monsoon*100:.0f}%", f"₹{price:,.2f}", signal, forecast_str, log, analysis
    )

# Gradio Interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🚜 Autonomous AI Farm & Predictive Mandi Simulation")
    gr.Markdown("Real-time reinforcement learning, 7-day time-series market forecasting, and LLM post-season evaluation.")
    
    with gr.Row():
        mon_box = gr.Textbox(label="🌧️ Monsoon Index")
        prc_box = gr.Textbox(label="📈 Mandi Spot Price")
        sig_box = gr.Textbox(label="📡 7-Day Market Trend Signal")
        
    fc_box = gr.Textbox(label="📊 7-Day Price Forecast (Days t+1 to t+7)", value="Computing baseline...")

    with gr.Row():
        with gr.Column(variant="panel"):
            gr.Markdown("### 🤖 Agent 1 (Primary AI Farm)")
            a1_c = gr.Textbox(label="Balance", value="₹1,00,000")
            with gr.Row():
                a1_i = gr.Textbox(label="Crop Inv", value="0 Q")
                a1_cat = gr.Textbox(label="Livestock", value="0 🐄")
                a1_lab = gr.Textbox(label="Active Labor", value="No")
            a1_p = gr.Textbox(label="Field Status", value="🟫 Barren")
            
            gr.Markdown("**Farm Operations:**")
            with gr.Row():
                b_w = gr.Button("Wheat 🌾 (-₹10k)", size="sm")
                b_m = gr.Button("Mustard 🌼 (-₹5k)", size="sm")
                b_s = gr.Button("Sugarcane 🎋 (-₹25k)", size="sm")
            with gr.Row():
                b_irr = gr.Button("Irrigate 💧", size="sm")
                b_harv = gr.Button("Harvest 🚜", size="sm")
                b_sell = gr.Button("Sell Crops 💰", size="sm", variant="primary")
            
            gr.Markdown("**Capital Expansion:**")
            with gr.Row():
                b_cow = gr.Button("Buy Cattle 🐄 (-₹40k)", variant="secondary")
                b_lab = gr.Button("Hire Labor 👨‍🌾 (-₹200/day)", variant="secondary")

        with gr.Column(variant="panel"):
            gr.Markdown("### 🥊 Agent 2 (Rival AI Farm)")
            a2_c = gr.Textbox(label="Balance", value="₹1,00,000")
            with gr.Row():
                a2_i = gr.Textbox(label="Crop Inv", value="0 Q")
                a2_cat = gr.Textbox(label="Livestock", value="0 🐄")
                a2_lab = gr.Textbox(label="Active Labor", value="No")
            a2_p = gr.Textbox(label="Field Status", value="🟫 Barren")
            gr.Markdown("*(Reacts dynamically to 7-day market signals)*")

    with gr.Row():
        b_auto = gr.Button("▶️ Simulate 1 Day (Both Auto)", variant="stop")
        
    with gr.Row():
        log_box = gr.Textbox(label="Daily Market Log")
        llm_box = gr.Textbox(label="🧠 Gemini Strategic Review", lines=4)

    outs = [a1_c, a1_i, a1_p, a1_cat, a1_lab, a2_c, a2_i, a2_p, a2_cat, a2_lab, mon_box, prc_box, sig_box, fc_box, log_box, llm_box]
    
    b_w.click(lambda: sim_step(1), outputs=outs)
    b_m.click(lambda: sim_step(2), outputs=outs)
    b_s.click(lambda: sim_step(3), outputs=outs)
    b_irr.click(lambda: sim_step(4), outputs=outs)
    b_harv.click(lambda: sim_step(5), outputs=outs)
    b_sell.click(lambda: sim_step(6), outputs=outs)
    b_cow.click(lambda: sim_step(7), outputs=outs)
    b_lab.click(lambda: sim_step(8), outputs=outs)
    b_auto.click(lambda: sim_step(None), outputs=outs)

demo.launch(inline=True, share=False)

In [ ]:
%pip install pyngrok

In [ ]:
import gymnasium as gym
import numpy as np
import gradio as gr
import os
from langchain_google_genai import ChatGoogleGenerativeAI

# ==========================================
# 0. SETUP LLM & ENVIRONMENT CONSTANTS
# ==========================================
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY", "")  # Set GOOGLE_API_KEY in your environment; never hardcode secrets.
try:
    llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.3)
except Exception:
    llm = None

CROP_DATA = {
    0: {"name": "Barren", "cost": 0, "water_need": 0, "growth_rate": 0, "yield": 0},
    1: {"name": "Wheat 🌾", "cost": 10000, "water_need": 0.2, "growth_rate": 0.20, "yield": 100},
    2: {"name": "Mustard 🌼", "cost": 5000, "water_need": 0.1, "growth_rate": 0.35, "yield": 60},
    3: {"name": "Sugarcane 🎋", "cost": 25000, "water_need": 0.4, "growth_rate": 0.10, "yield": 300},
}

# ==========================================
# 1. 7-DAY TIME-SERIES MARKET PREDICTOR
# ==========================================
class MandiMarketPredictor:
    """Rolling-window forecasting engine for Mandi spot prices."""
    def __init__(self, history_window=14):
        self.history_window = history_window
        self.price_history = []

    def record_price(self, price):
        self.price_history.append(price)
        if len(self.price_history) > 60:
            self.price_history.pop(0)

    def forecast_7_days(self, current_price, monsoon_index):
        if len(self.price_history) < 3:
            # Fallback baseline when starting
            return [current_price * (1.0 + 0.01 * i) for i in range(1, 8)], "NEUTRAL"

        # Calculate momentum slope over the recent window
        recent = self.price_history[-min(len(self.price_history), self.history_window):]
        x = np.arange(len(recent))
        slope, _ = np.polyfit(x, recent, 1)

        # 7-day projections combining drift, monsoon factor, and mean reversion
        forecasts = []
        base_target = 2500.0 * monsoon_index
        for day in range(1, 8):
            trend_val = current_price + (slope * day)
            reversion = 0.05 * (base_target - trend_val)
            pred = max(900.0, trend_val + reversion)
            forecasts.append(round(float(pred), 2))

        # Generate trading signal
        max_future_price = max(forecasts)
        if max_future_price > current_price * 1.08:
            signal = "🟢 BULLISH (Hold for higher Mandi prices)"
        elif min(forecasts) < current_price * 0.93:
            signal = "🔴 BEARISH (Sell now; price drop expected)"
        else:
            signal = "🟡 NEUTRAL (Market stable)"

        return forecasts, signal

# ==========================================
# 2. FULL MULTI-AGENT ENVIRONMENT
# ==========================================
class FullFarmEnv:
    def __init__(self):
        self.max_steps = 365
        self.predictor = MandiMarketPredictor()
        self.reset()
        # first class defined and then use of oops 

    def reset(self):
        self.current_step = 0
        self.monsoon = float(np.random.uniform(0.6, 1.0))
        self.base_demand = float(np.random.uniform(0.6, 1.0))
        self.market_history = []
        self.current_price = 2200.0
        self.predictor.price_history = []
        self.predictor.record_price(self.current_price)
        
        self.agents = {
            "Agent_1": {"cash": 100000.0, "inv": 0.0, "crop_type": 0, "crop_prog": 0.0, "cattle": 0, "labor": 0},
            "Agent_2": {"cash": 100000.0, "inv": 0.0, "crop_type": 0, "crop_prog": 0.0, "cattle": 0, "labor": 0}
        }
        return self.agents
        # Creation of agents 
    def step(self, act1, act2):
        self.current_step += 1
        self.monsoon = float(np.clip(self.monsoon + np.random.normal(0, 0.04), 0.2, 1.0))
        self.base_demand = float(np.clip(self.base_demand + np.random.normal(0, 0.05), 0.3, 1.0))
        actions = {"Agent_1": act1, "Agent_2": act2}
        total_supply_sold = 0.0
        # agents into action 

        for name, act in actions.items():
            a = self.agents[name]
            
            # Daily Passive Operations (Labor & Cattle)
            if a["labor"] > 0:
                if a["cash"] >= 200: a["cash"] -= 200
                else: a["labor"] = 0
            
            if a["cattle"] > 0:
                feed = a["cattle"] * 150
                milk = a["cattle"] * 400
                if a["cash"] >= feed:
                    a["cash"] += (milk - feed)
                else:
                    a["cattle"] = max(0, a["cattle"] - 1)

            # Active Decisions
            if act in [1, 2, 3] and a["crop_type"] == 0:
                cost = CROP_DATA[act]["cost"]
                if a["cash"] >= cost:
                    a["cash"] -= cost
                    a["crop_type"] = act
                    a["crop_prog"] = 0.1
                    
            elif act == 4 and a["crop_type"] != 0 and a["crop_prog"] < 1.0:
                if a["cash"] >= 2000:
                    a["cash"] -= 2000
                    efficiency = self.monsoon / CROP_DATA[a["crop_type"]]["water_need"]
                    mult = 1.5 if a["labor"] > 0 else 1.0
                    a["crop_prog"] = min(1.0, a["crop_prog"] + CROP_DATA[a["crop_type"]]["growth_rate"] * min(1.5, efficiency) * mult)
                    
            elif act == 5 and a["crop_prog"] >= 0.8:
                a["inv"] += CROP_DATA[a["crop_type"]]["yield"]
                a["crop_type"] = 0
                a["crop_prog"] = 0.0
                
            elif act == 6 and a["inv"] > 0:
                total_supply_sold += a["inv"]
                
            elif act == 7 and a["cash"] >= 40000:
                a["cash"] -= 40000
                a["cattle"] += 1
                self.market_history.append(f"Day {self.current_step}: {name} bought 1 Cow.")
                
            elif act == 8 and a["labor"] == 0:
                a["labor"] = 1
                self.market_history.append(f"Day {self.current_step}: {name} hired farm labor.")

        # Price Clearing
        oversupply = min(0.40, (total_supply_sold / 200.0) * 0.40)
        self.current_price = max(900.0, float(2500.0 * (self.base_demand - oversupply)))
        self.predictor.record_price(self.current_price)

        for name, act in actions.items():
            a = self.agents[name]
            if act == 6 and a["inv"] > 0:
                a["cash"] += (a["inv"] * self.current_price)
                self.market_history.append(f"Day {self.current_step}: {name} sold {a['inv']:.0f}Q at ₹{self.current_price:.0f}/Q.")
                a["inv"] = 0.0

        done = self.current_step >= self.max_steps
        return self.agents, self.current_price, done

env = FullFarmEnv()

# ==========================================
# 3. FORECAST-INFORMED AGENT POLICIES
# ==========================================
def rival_policy(s, signal):
    # Agent 2 uses market signal for selling
    if s["inv"] > 0 and "BEARISH" in signal:
        return 6 # Dump immediately if price crash predicted
    elif s["inv"] > 0 and "BULLISH" not in signal:
        return 6
    elif s["cattle"] == 0 and s["cash"] >= 45000: return 7 
    elif s["crop_type"] == 0 and s["cash"] >= 10000: return 1
    elif 0 < s["crop_prog"] < 0.8 and s["cash"] >= 2000: return 4
    elif s["crop_prog"] >= 0.8: return 5
    return 0

def format_stage(ctype, prog):
    if ctype == 0: return "🟫 Barren"
    return f"{CROP_DATA[ctype]['name']} ({prog*100:.0f}%)"

def get_llm_summary(history, c1, c2):
    if not llm: return "LLM not configured."
    prompt = f"""
    Review this 1-year agricultural market simulation:
    Agent 1 Balance: ₹{c1} | Agent 2 Balance: ₹{c2}
    Transaction Log: {history}
    
    Write a 3-sentence economic breakdown explaining how the 7-day Mandi price predictions, crop selection, and livestock investments influenced the final winner.
    """
    try:
        return llm.invoke(prompt).content
    except Exception:
        return "LLM Analysis unavailable."

# ==========================================
# 4. SIMULATION STEP & DASHBOARD
# ==========================================
def sim_step(act1):
    a1 = env.agents["Agent_1"]
    forecasts, signal = env.predictor.forecast_7_days(env.current_price, env.monsoon)
    
    # Auto-policy for Agent 1: Uses 7-day predictive timing for selling
    if act1 is None:
        if a1["inv"] > 0:
            act1 = 0 if "BULLISH" in signal else 6 # Hold if bullish, sell if bearish/neutral
        elif a1["cattle"] < 2 and a1["cash"] > 50000: act1 = 7
        elif a1["labor"] == 0 and a1["cash"] > 20000: act1 = 8
        elif a1["crop_prog"] >= 0.8: act1 = 5
        elif a1["crop_prog"] > 0: act1 = 4
        elif a1["crop_type"] == 0: act1 = 3 if a1["cash"] > 30000 else 2
        else: act1 = 0

    act2 = rival_policy(env.agents["Agent_2"], signal)
    states, price, done = env.step(act1, act2)
    s1, s2 = states["Agent_1"], states["Agent_2"]
    
    # Re-compute forecast after current day's price
    forecasts, signal = env.predictor.forecast_7_days(price, env.monsoon)
    forecast_str = " | ".join([f"D+{i+1}: ₹{p:.0f}" for i, p in enumerate(forecasts)])

    log = f"Day {env.current_step}/365 | Spot Price: ₹{price:,.2f}/Q"
    analysis = "Season in progress..."

    if done:
        log += f"\n\n🏆 SEASON COMPLETED! Agent 1: ₹{s1['cash']:,.0f} | Agent 2: ₹{s2['cash']:,.0f}"
        hist = "\n".join(env.market_history[-12:])
        analysis = get_llm_summary(hist, s1['cash'], s2['cash'])
        env.reset()

    return (
        f"₹{s1['cash']:,.2f}", f"{s1['inv']:.0f} Q", format_stage(s1['crop_type'], s1['crop_prog']), f"{s1['cattle']} 🐄", f"{'Yes 👨‍🌾' if s1['labor'] else 'No'}",
        f"₹{s2['cash']:,.2f}", f"{s2['inv']:.0f} Q", format_stage(s2['crop_type'], s2['crop_prog']), f"{s2['cattle']} 🐄", f"{'Yes 👨‍🌾' if s2['labor'] else 'No'}",
        f"{env.monsoon*100:.0f}%", f"₹{price:,.2f}", signal, forecast_str, log, analysis
    )

# Gradio Interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🚜 Autonomous AI Farm & Predictive Mandi Simulation")
    gr.Markdown("Real-time reinforcement learning, 7-day time-series market forecasting, and LLM post-season evaluation.")
    
    with gr.Row():
        mon_box = gr.Textbox(label="🌧️ Monsoon Index")
        prc_box = gr.Textbox(label="📈 Mandi Spot Price")
        sig_box = gr.Textbox(label="📡 7-Day Market Trend Signal")
        
    fc_box = gr.Textbox(label="📊 7-Day Price Forecast (Days t+1 to t+7)", value="Computing baseline...")

    with gr.Row():
        with gr.Column(variant="panel"):
            gr.Markdown("### 🤖 Agent 1 (Primary AI Farm)")
            a1_c = gr.Textbox(label="Balance", value="₹1,00,000")
            with gr.Row():
                a1_i = gr.Textbox(label="Crop Inv", value="0 Q")
                a1_cat = gr.Textbox(label="Livestock", value="0 🐄")
                a1_lab = gr.Textbox(label="Active Labor", value="No")
            a1_p = gr.Textbox(label="Field Status", value="🟫 Barren")
            
            gr.Markdown("**Farm Operations:**")
            with gr.Row():
                b_w = gr.Button("Wheat 🌾 (-₹10k)", size="sm")
                b_m = gr.Button("Mustard 🌼 (-₹5k)", size="sm")
                b_s = gr.Button("Sugarcane 🎋 (-₹25k)", size="sm")
            with gr.Row():
                b_irr = gr.Button("Irrigate 💧", size="sm")
                b_harv = gr.Button("Harvest 🚜", size="sm")
                b_sell = gr.Button("Sell Crops 💰", size="sm", variant="primary")
            
            gr.Markdown("**Capital Expansion:**")
            with gr.Row():
                b_cow = gr.Button("Buy Cattle 🐄 (-₹40k)", variant="secondary")
                b_lab = gr.Button("Hire Labor 👨‍🌾 (-₹200/day)", variant="secondary")

        with gr.Column(variant="panel"):
            gr.Markdown("### 🥊 Agent 2 (Rival AI Farm)")
            a2_c = gr.Textbox(label="Balance", value="₹1,00,000")
            with gr.Row():
                a2_i = gr.Textbox(label="Crop Inv", value="0 Q")
                a2_cat = gr.Textbox(label="Livestock", value="0 🐄")
                a2_lab = gr.Textbox(label="Active Labor", value="No")
            a2_p = gr.Textbox(label="Field Status", value="🟫 Barren")
            gr.Markdown("*(Reacts dynamically to 7-day market signals)*")

    with gr.Row():
        b_auto = gr.Button("▶️ Simulate 1 Day (Both Auto)", variant="stop")
        
    with gr.Row():
        log_box = gr.Textbox(label="Daily Market Log")
        llm_box = gr.Textbox(label="🧠 Gemini Strategic Review", lines=4)

    outs = [a1_c, a1_i, a1_p, a1_cat, a1_lab, a2_c, a2_i, a2_p, a2_cat, a2_lab, mon_box, prc_box, sig_box, fc_box, log_box, llm_box]
    
    b_w.click(lambda: sim_step(1), outputs=outs)
    b_m.click(lambda: sim_step(2), outputs=outs)
    b_s.click(lambda: sim_step(3), outputs=outs)
    b_irr.click(lambda: sim_step(4), outputs=outs)
    b_harv.click(lambda: sim_step(5), outputs=outs)
    b_sell.click(lambda: sim_step(6), outputs=outs)
    b_cow.click(lambda: sim_step(7), outputs=outs)
    b_lab.click(lambda: sim_step(8), outputs=outs)
    b_auto.click(lambda: sim_step(None), outputs=outs)

demo.launch(inline=True, share=False)

In [ ]:
%pip install pyngrok

In [ ]:
import gymnasium as gym
import numpy as np
import gradio as gr
import os
from langchain_google_genai import ChatGoogleGenerativeAI

# ==========================================
# 0. SETUP LLM & CONSTANTS
# ==========================================
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY", "")  # Set GOOGLE_API_KEY in your environment; never hardcode secrets.
try:
    llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.3)
except Exception:
    llm = None

CROP_DATA = {
    0: {"name": "Barren", "cost": 0, "water_need": 0, "growth_rate": 0, "yield": 0},
    1: {"name": "Wheat 🌾", "cost": 10000, "water_need": 0.2, "growth_rate": 0.20, "yield": 100},
    2: {"name": "Mustard 🌼", "cost": 5000, "water_need": 0.1, "growth_rate": 0.35, "yield": 60},
    3: {"name": "Sugarcane 🎋", "cost": 25000, "water_need": 0.4, "growth_rate": 0.10, "yield": 300},
}

# ==========================================
# 1. 7-DAY MARKET PREDICTOR
# ==========================================
class MandiMarketPredictor:
    def __init__(self, history_window=14):
        self.history_window = history_window
        self.price_history = []

    def record_price(self, price):
        self.price_history.append(price)
        if len(self.price_history) > 60: self.price_history.pop(0)

    def forecast_7_days(self, current_price, monsoon_index):
        if len(self.price_history) < 3:
            return [current_price * (1.0 + 0.01 * i) for i in range(1, 8)], "🟡 NEUTRAL (Market stable)"
        
        recent = self.price_history[-min(len(self.price_history), self.history_window):]
        slope, _ = np.polyfit(np.arange(len(recent)), recent, 1)

        forecasts = []
        base_target = 2500.0 * monsoon_index
        for day in range(1, 8):
            trend_val = current_price + (slope * day)
            pred = max(900.0, trend_val + 0.05 * (base_target - trend_val))
            forecasts.append(round(float(pred), 2))

        if max(forecasts) > current_price * 1.08: signal = "🟢 BULLISH (Hold for higher prices)"
        elif min(forecasts) < current_price * 0.93: signal = "🔴 BEARISH (Sell now; crash expected)"
        else: signal = "🟡 NEUTRAL (Market stable)"
        return forecasts, signal

# ==========================================
# 2. FULL HIERARCHICAL ENVIRONMENT
# ==========================================
class MasterFarmEnv:
    def __init__(self):
        self.max_steps = 365  
        self.predictor = MandiMarketPredictor()
        self.reset() 

    def reset(self):
        self.current_step = 0
        self.monsoon = float(np.random.uniform(0.6, 1.0))
        self.base_demand = float(np.random.uniform(0.6, 1.0))
        self.market_history = []
        self.current_price = 2200.0
        self.predictor.price_history = []
        self.predictor.record_price(self.current_price)
        
        # Added Machinery State
        agent_state = {"cash": 100000.0, "inv": 0.0, "crop_type": 0, "crop_prog": 0.0, "cattle": 0, "labor": 0, "pump": False, "tractor": False}
        self.agents = {"Agent_1": agent_state.copy(), "Agent_2": agent_state.copy()}
        return self.agents

    def step(self, act1, act2):
        self.current_step += 1
        self.monsoon = float(np.clip(self.monsoon + np.random.normal(0, 0.04), 0.2, 1.0))
        self.base_demand = float(np.clip(self.base_demand + np.random.normal(0, 0.05), 0.3, 1.0))
        actions = {"Agent_1": act1, "Agent_2": act2}
        total_sold = 0.0

        for name, act in actions.items():
            a = self.agents[name]
            
            # Passive Operations
            if a["labor"] > 0:
                if a["cash"] >= 200: a["cash"] -= 200
                else: a["labor"] = 0
            if a["cattle"] > 0:
                if a["cash"] >= (a["cattle"] * 150): a["cash"] += (a["cattle"] * 250)
                else: a["cattle"] = max(0, a["cattle"] - 1)

            # Active Operations (1-3: Plant, 4: Irrigate, 5: Harvest, 6: Sell)
            if act in [1, 2, 3] and a["crop_type"] == 0 and a["cash"] >= CROP_DATA[act]["cost"]:
                a["cash"] -= CROP_DATA[act]["cost"]
                a["crop_type"] = act
                a["crop_prog"] = 0.1
                    
            elif act == 4 and a["crop_type"] != 0 and a["crop_prog"] < 1.0:
                water_cost = 0 if a["pump"] else 2000
                if a["cash"] >= water_cost:
                    a["cash"] -= water_cost
                    eff = self.monsoon / CROP_DATA[a["crop_type"]]["water_need"]
                    # Tractor doubles speed, labor adds 50%
                    mult = 2.0 if a["tractor"] else (1.5 if a["labor"] > 0 else 1.0)
                    a["crop_prog"] = min(1.0, a["crop_prog"] + CROP_DATA[a["crop_type"]]["growth_rate"] * min(1.5, eff) * mult)
                    
            elif act == 5 and a["crop_prog"] >= 0.8:
                a["inv"] += CROP_DATA[a["crop_type"]]["yield"]
                a["crop_type"] = 0
                a["crop_prog"] = 0.0
                
            elif act == 6 and a["inv"] > 0:
                total_sold += a["inv"]
                
            # Capital Investments (7: Cattle, 8: Labor, 9: Pump, 10: Tractor)
            elif act == 7 and a["cash"] >= 40000:
                a["cash"] -= 40000; a["cattle"] += 1
            elif act == 8 and a["labor"] == 0:
                a["labor"] = 1
            elif act == 9 and a["cash"] >= 30000 and not a["pump"]:
                a["cash"] -= 30000; a["pump"] = True
                self.market_history.append(f"Day {self.current_step}: {name} bought Solar Pump.")
            elif act == 10 and a["cash"] >= 80000 and not a["tractor"]:
                a["cash"] -= 80000; a["tractor"] = True
                self.market_history.append(f"Day {self.current_step}: {name} bought Tractor.")

        # Price Clearing
        oversupply = min(0.40, (total_sold / 200.0) * 0.40)
        self.current_price = max(900.0, float(2500.0 * (self.base_demand - oversupply)))
        self.predictor.record_price(self.current_price)

        for name, act in actions.items():
            if act == 6 and self.agents[name]["inv"] > 0:
                self.agents[name]["cash"] += (self.agents[name]["inv"] * self.current_price)
                self.market_history.append(f"Day {self.current_step}: {name} sold {self.agents[name]['inv']:.0f}Q at ₹{self.current_price:.0f}/Q.")
                self.agents[name]["inv"] = 0.0

        return self.agents, self.current_price, self.current_step >= self.max_steps

env = MasterFarmEnv()

# ==========================================
# 3. HIERARCHICAL POLICIES
# ==========================================
def master_ai_policy(s, signal):
    """3-Layer Hierarchical Logic for Agent 1"""
    # LAYER 1: Chief Financial Agent (CapEx)
    if not s["tractor"] and s["cash"] >= 100000: return 10
    if not s["pump"] and s["cash"] >= 50000: return 9
    if s["cattle"] < 2 and s["cash"] >= 60000: return 7
    if s["labor"] == 0 and s["cash"] >= 20000 and not s["tractor"]: return 8

    # LAYER 2: Trading Agent (Market Timing)
    if s["inv"] > 0:
        return 0 if "BULLISH" in signal else 6

    # LAYER 3: Agronomy Agent (Farming Loop)
    if s["crop_prog"] >= 0.8: return 5
    if s["crop_prog"] > 0: return 4
    if s["crop_type"] == 0: return 3 if s["cash"] > 30000 else 2
    return 0

def rival_policy(s, signal):
    if s["inv"] > 0 and "BEARISH" in signal: return 6
    elif s["inv"] > 0 and "BULLISH" not in signal: return 6
    elif s["cattle"] == 0 and s["cash"] >= 45000: return 7 
    elif s["crop_type"] == 0 and s["cash"] >= 10000: return 1
    elif 0 < s["crop_prog"] < 0.8 and s["cash"] >= 2000: return 4
    elif s["crop_prog"] >= 0.8: return 5
    return 0

# ==========================================
# 4. DASHBOARD & EXECUTION
# ==========================================
def format_stage(c, p): return "🟫 Barren" if c == 0 else f"{CROP_DATA[c]['name']} ({p*100:.0f}%)"

def sim_step(manual_act):
    forecasts, signal = env.predictor.forecast_7_days(env.current_price, env.monsoon)
    act1 = master_ai_policy(env.agents["Agent_1"], signal) if manual_act is None else manual_act
    act2 = rival_policy(env.agents["Agent_2"], signal)
    
    states, price, done = env.step(act1, act2)
    s1, s2 = states["Agent_1"], states["Agent_2"]
    
    forecasts, signal = env.predictor.forecast_7_days(price, env.monsoon)
    
    log = f"Day {env.current_step}/365 | Spot Price: ₹{price:,.2f}/Q"
    analysis = "Season running..."

    if done:
        log += f"\n\n🏆 SEASON OVER! A1: ₹{s1['cash']:,.0f} | A2: ₹{s2['cash']:,.0f}"
        if llm:
            prompt = f"Analyze this 1-year AI farming sim. A1 Cash: ₹{s1['cash']} | A2 Cash: ₹{s2['cash']}. Log: {' | '.join(env.market_history[-10:])}. Summarize in 3 sentences how machinery investments and market timing decided the winner."
            try: analysis = llm.invoke(prompt).content
            except: analysis = "LLM Generation Failed."
        env.reset()

    return (
        f"₹{s1['cash']:,.2f}", f"{s1['inv']:.0f} Q", format_stage(s1['crop_type'], s1['crop_prog']), 
        f"Pump: {'✅' if s1['pump'] else '❌'} | Tractor: {'✅' if s1['tractor'] else '❌'}",
        f"₹{s2['cash']:,.2f}", f"{s2['inv']:.0f} Q", format_stage(s2['crop_type'], s2['crop_prog']), 
        f"Pump: {'✅' if s2['pump'] else '❌'} | Tractor: {'✅' if s2['tractor'] else '❌'}",
        f"{env.monsoon*100:.0f}%", f"₹{price:,.2f}", signal, log, analysis
    )

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🚜 Master AI Farm Architecture: CapEx & Hierarchical Planning")
    
    with gr.Row():
        mon_box = gr.Textbox(label="🌧️ Monsoon Index")
        prc_box = gr.Textbox(label="📈 Mandi Price")
        sig_box = gr.Textbox(label="📡 7-Day Market Trend")

    with gr.Row():
        with gr.Column(variant="panel"):
            gr.Markdown("### 🤖 Agent 1 (Hierarchical AI)")
            a1_c = gr.Textbox(label="Balance", value="₹1,00,000")
            a1_i = gr.Textbox(label="Godown Inventory", value="0 Q")
            a1_p = gr.Textbox(label="Field Status", value="🟫 Barren")
            a1_mach = gr.Textbox(label="Machinery", value="Pump: ❌ | Tractor: ❌")
            
            with gr.Row():
                b_w = gr.Button("Plant Wheat (-₹10k)", size="sm")
                b_s = gr.Button("Plant Cane (-₹25k)", size="sm")
                b_irr = gr.Button("Irrigate 💧", size="sm")
            with gr.Row():
                b_harv = gr.Button("Harvest 🚜", size="sm")
                b_sell = gr.Button("Sell 💰", size="sm", variant="primary")
                b_pump = gr.Button("Buy Pump (-₹30k)", size="sm")
                b_trac = gr.Button("Buy Tractor (-₹80k)", size="sm")

        with gr.Column(variant="panel"):
            gr.Markdown("### 🥊 Agent 2 (Standard AI)")
            a2_c = gr.Textbox(label="Balance", value="₹1,00,000")
            a2_i = gr.Textbox(label="Godown Inventory", value="0 Q")
            a2_p = gr.Textbox(label="Field Status", value="🟫 Barren")
            a2_mach = gr.Textbox(label="Machinery", value="Pump: ❌ | Tractor: ❌")

    with gr.Row():
        b_auto = gr.Button("▶️ Simulate 1 Day (Auto-Run)", variant="stop")
        
    with gr.Row():
        log_box = gr.Textbox(label="Market Feed")
        llm_box = gr.Textbox(label="🧠 Executive AI Summary", lines=3)

    outs = [a1_c, a1_i, a1_p, a1_mach, a2_c, a2_i, a2_p, a2_mach, mon_box, prc_box, sig_box, log_box, llm_box]
    
    b_w.click(lambda: sim_step(1), outputs=outs); b_s.click(lambda: sim_step(3), outputs=outs)
    b_irr.click(lambda: sim_step(4), outputs=outs); b_harv.click(lambda: sim_step(5), outputs=outs)
    b_sell.click(lambda: sim_step(6), outputs=outs); b_pump.click(lambda: sim_step(9), outputs=outs)
    b_trac.click(lambda: sim_step(10), outputs=outs); b_auto.click(lambda: sim_step(None), outputs=outs)
    from pyngrok import ngrok

# Paste your copied authtoken inside the quotes
ngrok.set_auth_token("3IMmQTHdyczUdOaP9qkKJ1HIWkD_6RcCLwqKkfZW2RuxTJs49")

# Reset any existing tunnels
ngrok.kill()

# Create public tunnel for Gradio port
public_url = ngrok.connect(7865)
print(f"🔥 YOUR PUBLIC LIVE LINK: {public_url}")

# Launch Gradio
demo.launch(share=False)

